# Análisis Exploratorio
Melisa Mendizabal - Belen Monterroso - Renato Rojas

## Carga, armonización y calidad de datos

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Lab7_ENEIC")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

mydir = "/opt/app/working_dir/lab7/"

COLUMNAS_REQUERIDAS = [
    "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE",
    "OCUPADOS", "P05C16", "P05D01",
    "P02A03", "P05C07A", "P05C07B", "P05H01A",
    "P03A03A", "DOMINIO",
]

# archivo: (nombre_excel, periodo_archivo, anio_archivo, trimestre_calendario)
ARCHIVOS_2025 = {
    "2025T1": ("personas_2025_t1.xlsx", 2025, 1),
    "2025T2": ("personas_2025_t2.xlsx", 2025, 2),
    "2025T3": ("personas_2025_t3.xlsx", 2025, 3),
    "2025T4": ("personas_2025_t4.xlsx", 2025, 4),
}
ARCHIVO_2026 = {"2026T1": ("personas_2026_t1.xlsx", 2026, 1)}

def excel_a_parquet(nombre_excel, periodo_archivo, anio_archivo, trimestre_cal):
    ruta_in = os.path.join(mydir, nombre_excel)
    df_pd = pd.read_excel(ruta_in)
    df_pd.columns = [c.strip().upper() for c in df_pd.columns]
    df_pd = df_pd[[c for c in COLUMNAS_REQUERIDAS if c in df_pd.columns]]

    # Homologar: un mismo código puede llegar como número o texto -> forzar a texto
    for c in ["P05C16", "P03A03A", "DOMINIO", "OCUPADOS"]:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].apply(lambda v: None if pd.isna(v) else str(v).strip())

    for c in ["NUM_HOGAR", "NUM_PERSONA", "ANIO", "TRIMESTRE"]:
        if c in df_pd.columns:
            df_pd[c] = pd.to_numeric(df_pd[c], errors="coerce")

    for c in ["P02A03", "P05C07A", "P05C07B", "P05H01A", "P05D01", "FACTOR"]:
        if c in df_pd.columns:
            df_pd[c] = pd.to_numeric(df_pd[c], errors="coerce")

    df_pd["archivo_origen"] = nombre_excel
    df_pd["periodo_archivo"] = periodo_archivo
    df_pd["anio_archivo"] = anio_archivo
    df_pd["trimestre_calendario"] = trimestre_cal

    ruta_out = os.path.join(mydir, "raw_parquet", f"{periodo_archivo}.parquet")
    os.makedirs(os.path.dirname(ruta_out), exist_ok=True)
    df_pd.to_parquet(ruta_out, index=False)
    return ruta_out

In [ ]:
rutas_2025 = [excel_a_parquet(nom, per, an, tc) for per,(nom,an,tc) in ARCHIVOS_2025.items()]
dfs_2025 = [spark.read.parquet(r) for r in rutas_2025]

df_2025_raw = dfs_2025[0]
for df in dfs_2025[1:]:
    df_2025_raw = df_2025_raw.unionByName(df, allowMissingColumns=True)

df_2025_raw.printSchema()
df_2025_raw.show(5, truncate=False)

# Registros por archivo, antes de filtros
df_2025_raw.groupBy("archivo_origen").count().orderBy("archivo_origen").show()

In [ ]:
def preparar(df, poblacion_nombre):
    df = (
        df
        .withColumn("edad", F.col("P02A03").cast("double"))
        .withColumn("antiguedad_anios", F.col("P05C07A").cast("double"))
        .withColumn("antiguedad_meses", F.col("P05C07B").cast("double"))
        .withColumn("horas_semanales", F.col("P05H01A").cast("double"))
        .withColumn("salario_mensual", F.col("P05D01").cast("double"))
        .withColumn("ocupado", F.col("OCUPADOS").cast("int"))
        .withColumn("categoria_ocupacional", F.trim(F.col("P05C16")))
        .withColumn(
            "nivel_educativo",
            F.when(F.trim(F.col("P03A03A")) == "0", "0")          # 0 = "ninguno", NO es faltante
             .when(F.col("P03A03A").isNull(), "DESCONOCIDO")
             .otherwise(F.trim(F.col("P03A03A")))
        )
        .withColumn("dominio", F.when(F.col("DOMINIO").isNull(), "DESCONOCIDO").otherwise(F.trim(F.col("DOMINIO"))))
        .withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses")/F.lit(12))
    )

    pasos = {}
    pasos["0_inicial"] = df.count()

    df = df.filter(F.col("ocupado") == 1)
    pasos["1_ocupados"] = df.count()

    df = df.filter(F.col("categoria_ocupacional").isin("1","2","3","4"))
    pasos["2_asalariados"] = df.count()

    df = df.filter(F.col("salario_mensual").isNotNull() & ~F.isnan("salario_mensual") & (F.col("salario_mensual") > 0))
    pasos["3_salario_valido"] = df.count()

    df = df.filter(F.col("edad").isNotNull() & ~F.isnan("edad") & (F.col("edad") >= 15))
    pasos["4_edad_valida"] = df.count()

    df = df.filter(F.col("antiguedad_meses").isNotNull() & (F.col("antiguedad_meses") >= 0) & (F.col("antiguedad_meses") <= 11))
    pasos["5_meses_validos"] = df.count()

    df = df.filter(F.col("antiguedad").isNotNull() & (F.col("antiguedad") >= 0))
    pasos["6_antiguedad_no_negativa"] = df.count()

    df = df.filter(F.col("antiguedad") <= F.col("edad"))
    pasos["7_antiguedad_menor_igual_edad"] = df.count()

    df = df.filter(F.col("horas_semanales").isNotNull() & (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168))
    pasos["8_horas_validas"] = df.count()

    print(f"--- Exclusión paso a paso: {poblacion_nombre} ---")
    for k, v in pasos.items():
        print(k, v)

    return df

df_2025_final = preparar(df_2025_raw, "2025")